Dataset analysis: https://www.kaggle.com/datasets/aouatifcherdid/healthcare-dataset-stroke-data

##ENG. BELOW

This project is an end-to-end analysis of the "Healthcare Dataset Stroke Data" using SQL (DuckDB). The goal of the project was to clean the raw data, prepare it for analysis, and investigate which demographic and health factors have the greatest impact on the occurrence of a stroke in patients.

**Tools**:


DuckDB, PyCharm(DataGrip)


In [ ]:
%%sql
CREATE TABLE baza_serce AS
SELECT *
FROM 'Dane/healthcare_dataset_stroke_data.csv';

In [2]:
%%sql
SELECT *
FROM baza_serce
LIMIT 100;

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67,0,1,True,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61,0,0,True,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80,0,1,True,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49,0,0,True,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79,1,0,True,Self-employed,Rural,174.12,24.0,never smoked,1
...,...,...,...,...,...,...,...,...,...,...,...,...
95,2458,Female,78,0,0,True,Private,Rural,235.63,32.3,never smoked,1
96,35512,Female,70,0,0,True,Self-employed,Rural,76.34,24.4,formerly smoked,1
97,56841,Male,58,0,1,True,Private,Rural,240.59,31.4,smokes,1
98,8154,Male,57,1,0,True,Govt_job,Urban,78.92,27.7,formerly smoked,1


ENG. BELOW. Before starting the analysis, I check the quality of the raw data. I look for outliers, hidden missing data (saved as text), and inconsistencies in categorical variables

Outliers verification

In [3]:
%%sql
SELECT
MIN(age) AS min_age,
MAX(age) AS max_age,
MIN(avg_glucose_level) AS min_glucose,
MAX(avg_glucose_level) AS max_glucose
FROM baza_serce;

,min_age,max_age,min_glucose,max_glucose
0,0.08,82,55.12,271.74


Identification of pseudo-missing values

In [4]:
%%sql
SELECT
    COUNT(*) AS liczba_wierszy,
    SUM(CASE WHEN bmi = 'N/A' THEN 1 ELSE 0 END) AS bmi_puste,
    SUM(CASE WHEN smoking_status = 'Unknown' THEN 1 ELSE 0 END) AS palenie_puste
FROM baza_serce;

,liczba_wierszy,bmi_puste,palenie_puste
0,5110,201,1544


Categorical variables check

In [6]:
%%sql
SELECT gender,
    COUNT(*) AS lista_plec
FROM baza_serce
GROUP BY gender;

,gender,lista_plec
0,Other,1
1,Female,2994
2,Male,2115


Based on the profiling insights, I create a new, optimized table. The main task is to convert pseudo-missing data ('N/A') into system NULL values, which guarantees the correctness of subsequent mathematical operations.

In [ ]:
%%sql
CREATE TABLE cleaned_baza_serce AS
    SELECT
id,
CASE
    WHEN gender = 'Other' THEN NULL
    ELSE gender
END AS gender,
age,
hypertension,
heart_disease,
ever_married,
work_tyPe,
Residence_type,
avg_glucose_level,
NULLIF(bmi, 'N/A') AS bmi,
NULLIF(smoking_status, 'Unknown') AS smoking_status,
stroke
FROM baza_serce;

On the cleaned dataset, I perform an analysis to identify the main stroke risk factors. I use aggregations, grouping, and conditional logic for data transformation.

I check what percentage of the total population in the sample had a stroke. // Ile mamy danych po czyszczeniu i jaki % ma udar?


In [10]:
%%sql
SELECT
    COUNT (*) AS wszyscy_pacjenci,
    SUM(stroke) AS udary,
    ROUND((SUM(stroke) *100)/ COUNT(*), 2) AS procent_udarow
   FROM cleaned_baza_serce;

,wszyscy_pacjenci,udary,procent_udarow
0,5110,249,4.87


I transform the continuous variable (age) into age categories (categorical variable) using the CASE WHEN clause. // Kto najczęściej ma udar? - wiek

In [13]:
%%sql
SELECT
    CASE
    WHEN age < 40 THEN '1. Under 40 years'
    WHEN age>= 40 AND age<= 60 THEN '2. 40-60 years'
    ELSE '3. Over 60 years'
    END AS age_grupy,
    COUNT (*) AS wszyscy_pacjenci,
    SUM(stroke) AS udary,
    ROUND((SUM(stroke) *100)/ COUNT(*), 2) AS procent_udarow
FROM cleaned_baza_serce
GROUP BY age_grupy
ORDER BY age_grupy;

,age_grupy,wszyscy_pacjenci,udary,procent_udarow
0,1. Under 40 years,2170,8,0.37
1,2. 40-60 years,1636,64,3.91
2,3. Over 60 years,1304,177,13.57


I cross-check how the occurrence of two different conditions (hypertension and heart disease) drastically increases the stroke rate. // Choroby serca i nadciśnienie a ryzyko udaru

In [16]:
%%sql
SELECT
    hypertension AS nadcisnienie,
    heart_disease AS choroby_serca,
    COUNT(*) AS wszyscy_pacjenci,
    SUM(stroke) AS suma_udarow,
    ROUND((SUM(stroke) *100) / COUNT(*), 2) AS udar_zagrozenie
FROM cleaned_baza_serce
GROUP BY nadcisnienie, choroby_serca
ORDER BY udar_zagrozenie DESC;


,nadcisnienie,choroby_serca,wszyscy_pacjenci,suma_udarow,udar_zagrozenie
0,1,1,64,13,20.31
1,0,1,212,34,16.04
2,1,0,434,53,12.21
3,0,0,4400,149,3.39


I analyze the glucose level in smokers, filtering the dataset strictly to patients who had a stroke. It is crucial here to exclude records with missing data (IS NOT NULL) so as not to distort the aggregation results. // Odniesienie palenia do przebytych udarów.

In [20]:
%%sql
SELECT
    smoking_status AS czy_pali,
    COUNT(*) AS liczba_pacj_udar,
    ROUND(AVG(avg_glucose_level), 2) AS srednia_glukoza
FROM cleaned_baza_serce
WHERE
    stroke = 1
    AND czy_pali IS NOT NULL
GROUP BY czy_pali
ORDER BY srednia_glukoza DESC;

,czy_pali,liczba_pacj_udar,srednia_glukoza
0,smokes,42,139.99
1,never smoked,90,132.91
2,formerly smoked,70,132.81
